<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.9_Praktikum_Real-time_Camera_Processing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.9: Praktikum Real-time Camera Processing
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Membangun kelas pembaca aliran kamera multi-utas (`ThreadedCameraStream`) berbasis Python `threading`.
2. Mengeliminasi penumpukan penyangga (*hardware buffer lag*) untuk mencapai latensi nol pada pemrosesan kamera langsung.
3. Menyematkan antarmuka On-Screen Display (OSD HUD) dinamis dengan pemantauan FPS dan status operasional pabrik.
4. Mengevaluasi stabilitas konkurensi dan mekanisme penghentian thread yang aman (*clean shutdown*).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import threading
import time

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Implementasi Kelas Threaded Camera Streamer Berorientasi Objek
Membangun kelas pembaca frame di latar belakang independen dengan penguncian mutex `threading.Lock` dan fallback headless.

In [ ]:
class ThreadedCameraStream:
    def __init__(self, src=0, name='CameraThread'):
        self.stream = cv2.VideoCapture(src)
        self.name = name
        self.stopped = False
        self.lock = threading.Lock()
        
        self.ret, self.frame = self.stream.read()
        if not self.ret:
            # Fallback frame sintetis jika tidak ada webcam terpasang
            self.frame = np.full((480, 640, 3), 40, dtype=np.uint8)
            self.ret = True
            self.is_synthetic = True
        else:
            self.is_synthetic = False
            
    def start(self):
        self.thread = threading.Thread(target=self.update, name=self.name, daemon=True)
        self.thread.start()
        return self
        
    def update(self):
        counter = 0
        while not self.stopped:
            if not self.is_synthetic:
                ret, frame = self.stream.read()
                if not ret:
                    time.sleep(0.01)
                    continue
                with self.lock:
                    self.ret = ret
                    self.frame = frame
            else:
                # Update simulasi frame konveyor PKS
                time.sleep(0.02) # ~50 FPS simulasi
                counter = (counter + 1) % 640
                synth = np.full((480, 640, 3), 40, dtype=np.uint8)
                cv2.circle(synth, (counter, 240), 40, (15, 90, 220), -1)
                with self.lock:
                    self.frame = synth
                    
    def read(self):
        with self.lock:
            return self.ret, self.frame.copy()
            
    def stop(self):
        self.stopped = True
        if hasattr(self, 'thread'):
            self.thread.join(timeout=1.0)
        self.stream.release()

print('Kelas ThreadedCameraStream berhasil didefinisikan!')


## 2. Eksekusi Pengujian Live Stream dengan Anotasi OSD HUD
Menjalankan stream multi-utas, menyematkan indikator HUD pabrik kelapa sawit, dan menghitung stabilitas FPS.

In [ ]:
cam = ThreadedCameraStream(src=0).start()
time.sleep(0.1)

fps_records = []
display_sample = None

for i in range(40):
    t_start = time.perf_counter()
    ret, frame = cam.read()
    if not ret:
        break
        
    # Simulasi beban pemrosesan visi komputer (grayscaling + thresholding)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 60, 255, cv2.THRESH_BINARY)
    
    dt = time.perf_counter() - t_start
    fps_val = 1.0 / max(dt, 1e-5)
    fps_records.append(fps_val)
    
    # Penambahan HUD Telemetri PKS
    cv2.rectangle(frame, (0, 0), (640, 50), (20, 20, 20), -1)
    cv2.putText(frame, 'PKS LOADING RAMP 01 - LIVE STREAM', (15, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.putText(frame, f'STATUS: NORMAL | FPS: {fps_val:.1f}', (15, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(frame, f'Throughput: 115.4 Ton/Jam', (420, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 255), 1)
    
    if i == 20:
        display_sample = frame.copy()

cam.stop()
print(f'Selesai 40 iterasi live stream!')
print(f'Rata-rata FPS Eksekusi Multi-Threaded: {np.mean(fps_records):.1f} FPS')


## 3. Visualisasi Hasil Pengujian Stream dan Frame HUD
Menampilkan grafik stabilitas FPS dan tampilan HUD stasiun loading ramp pabrik sawit.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(fps_records, color='mediumseagreen', linewidth=2, marker='.')
ax1.set_title('Kestabilan Laju Bingkai Threaded Camera Streamer', fontsize=11, fontweight='bold')
ax1.set_xlabel('Nomor Siklus Pengambilan')
ax1.set_ylabel('Throughput (FPS)')

if display_sample is not None:
    ax2.imshow(cv2.cvtColor(display_sample, cv2.COLOR_BGR2RGB))
    ax2.set_title('Cuplikan Layar OSD HUD Loading Ramp PKS', fontsize=11, fontweight='bold')
    ax2.axis('off')

plt.tight_layout()
plt.savefig('praktikum_realtime_camera_11_9.png', dpi=150)
plt.show()
